# Phutball AlphaZero - TPU Training

Runtime → TPU v6e → Run all

In [ ]:
# Detect environment
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ or 'google.colab' in str(globals())
print(f"Running in Colab: {IN_COLAB}")

Running in Colab: True


In [ ]:
# Install dependencies
if IN_COLAB:
    # Try TPU first, fall back to GPU/CPU
    try:
        !pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
    except:
        !pip install -q jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    !pip install -q flax optax wandb mctx

In [ ]:
# Verify devices
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Device count: {jax.device_count()}")
print(f"Local device count: {jax.local_device_count()}")

DEVICE_COUNT = jax.device_count()
DEVICE_TYPE = str(jax.devices()[0]).split(':')[0] if jax.devices() else 'cpu'
print(f"\nUsing {DEVICE_COUNT}x {DEVICE_TYPE}")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX version: 0.7.2
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Device count: 1
Local device count: 1

Using 1x TPU_0(process=0,(0,0,0,0))


In [ ]:
# Mount Google Drive for checkpoints (Colab only)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_BASE = "/content/drive/MyDrive/phutball_checkpoints/v3"
else:
    CHECKPOINT_BASE = "./checkpoints"

os.makedirs(CHECKPOINT_BASE, exist_ok=True)
print(f"Checkpoints: {CHECKPOINT_BASE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints: /content/drive/MyDrive/phutball_checkpoints/v3


In [ ]:
# Wandb login (optional)
USE_WANDB = True

if USE_WANDB:
    import wandb
    wandb.login()

wandb: Currently logged in as: echoname6 (echoname6-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# Clone/update repo
REPO_URL = "https://github.com/echoname6/phutball-jax.git"
REPO_DIR = "/content/phutball-jax" if IN_COLAB else "./phutball-jax"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -3

remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (2/2), done.
remote: Total 3 (delta 2), reused 2 (delta 2), pack-reused 1 (from 1)
Unpacking objects: 100% (3/3), 462 bytes | 231.00 KiB/s, done.
From https://github.com/echoname6/phutball-jax
   e77633b..38a5ae2  main       -> origin/main
Updating e77633b..38a5ae2
Fast-forward
 train_batched.py | 17 ++++++++++++-----
 1 file changed, 12 insertions(+), 5 deletions(-)
/content/phutball-jax
38a5ae2 (HEAD -> main, origin/main, origin/HEAD) add new metrics to training print and wandb logging
e77633b fix league pool checkpoint format to include batch_stats
1ee1f45 add training diagnostics: KL divergence, policy entropy, value pred stats


In [ ]:
# Clear stale checkpoints from previous run (uncomment to reset)
#!rm -rf {CHECKPOINT_BASE}/standard

In [ ]:
# === TRAINING CONFIG ===

# Optimized for Gumbel MuZero on single TPU v6e
# See: "Policy improvement by planning with Gumbel" (ICLR 2022)
# Gumbel works well with 16-32 sims vs 100-800 for standard MCTS

BASE_GAMES_PER_DEVICE = 256  # Increased for better TPU utilization
BASE_TRAIN_BATCH_PER_DEVICE = 256

GAMES_BATCH_SIZE = max(256, BASE_GAMES_PER_DEVICE * DEVICE_COUNT)
TRAIN_BATCH_SIZE = max(256, BASE_TRAIN_BATCH_PER_DEVICE * DEVICE_COUNT)

print(f"Devices: {DEVICE_COUNT} | Games: {GAMES_BATCH_SIZE} | Train: {TRAIN_BATCH_SIZE}")

# Board stages: (rows, cols, iterations, mcts_sims)
# Using Gumbel-optimal sim counts (16-32 instead of 100-300)
BOARD_STAGES = [
    (11, 9, 50, 16),      # 16 sims for 11x9 (fast iteration)
    (15, 11, 150, 24),    # 24 sims for 15x11
    (21, 15, 300, 32),    # 32 sims for 21x15
]

NUM_CHANNELS = 128
NUM_RES_BLOCKS = 10
MAX_CONSIDERED_ACTIONS = 32  # Gumbel Sequential Halving needs this

# max_moves_per_game is computed automatically from board size (rows * cols * 2)

# Temperature schedule: explore for first N moves, then exploit
TEMP_THRESHOLD = 30          # Was 15, now 30 for more mid-game exploration
TEMP_FINAL = 0.1             # Small floor to prevent mode collapse

LEARNING_RATE = 0.002
WEIGHT_DECAY = 1e-4
TRAIN_STEPS_PER_ITER = 150
BUFFER_SIZE = 300000
MIN_BUFFER_SIZE = 1000

# N-jump curriculum (winning state generation)
CURRICULUM_ENABLED = True
CURRICULUM_INITIAL_RATIO = 0.6
CURRICULUM_FINAL_RATIO = 0.03
CURRICULUM_DECAY_ITERS = 100

# Random opponent mixing: adds diversity to break passive self-play equilibrium
# NOTE: Passive equilibria in earlier runs were likely caused by inverted winner detection
RANDOM_OPP_ENABLED = False
RANDOM_OPP_INITIAL_RATIO = 0.33   # Start with 33% games vs random
RANDOM_OPP_FINAL_RATIO = 0.003    # Decay to 0.3% by end
RANDOM_OPP_DECAY_ITERS = 100      # Iterations to decay

CHECKPOINT_EVERY = 5
EVAL_EVERY = 25

WANDB_PROJECT = "phutball-az-jax"

print(f"Total iterations: {sum(c[2] for c in BOARD_STAGES)}")
print(f"Temp schedule: 1.0 -> {TEMP_FINAL} after move {TEMP_THRESHOLD}")
print(f"Gumbel MCTS: {[s[3] for s in BOARD_STAGES]} sims per stage")
if RANDOM_OPP_ENABLED:
    print(f"Random opp: {RANDOM_OPP_INITIAL_RATIO:.0%} -> {RANDOM_OPP_FINAL_RATIO:.1%} over {RANDOM_OPP_DECAY_ITERS} iters")
else:
    print("Random opp: DISABLED (pure self-play)")

Devices: 1 | Games: 256 | Train: 256
Total iterations: 500
Temp schedule: 1.0 -> 0.1 after move 30
Gumbel MCTS: [16, 24, 32] sims per stage
Random opp: DISABLED (pure self-play)


In [ ]:
# Imports
import numpy as np
import time
from functools import partial

from phutball_env_jax import (
    PhutballState, EnvConfig, reset, step, get_legal_actions,
    state_to_network_input, render_board
)
from network import (
    PhutballNetwork, create_network, init_network,
    ChimeraNetwork, create_chimera_network, init_chimera_network,
)
from train_batched import (
    AlphaZeroTrainer, TrainConfig, make_train_config,
    ChimeraTrainer, ChimeraConfig,
)
from mcts import MCTSConfig, select_action

print("Imports OK")

Imports OK


In [ ]:
# Training mode: "standard", "chimera", or "overnight_11x9"
# overnight_11x9: Simple baseline to validate training loop (should hit 70%+ vs random)
TRAINING_MODE = "overnight_11x9"
CHIMERA_BOARD_SIZES = ((11, 9), (15, 11), (21, 15))

print(f"Training mode: {TRAINING_MODE}")

Training mode: overnight_11x9


In [ ]:
def create_pmap_train_config(
    rows: int,
    cols: int,
    num_iterations: int,
    num_simulations: int,
    checkpoint_dir: str,
    run_name: str = None,
) -> TrainConfig:
    """Create config optimized for multi-device training."""
    return make_train_config(
        rows=rows,
        cols=cols,
        num_iterations=num_iterations,
        num_simulations=num_simulations,

        # Scaled for parallelization
        batch_size_games=GAMES_BATCH_SIZE,
        batch_size_train=TRAIN_BATCH_SIZE,

        # Network
        num_channels=NUM_CHANNELS,
        num_res_blocks=NUM_RES_BLOCKS,

        # Training
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        train_steps_per_iteration=TRAIN_STEPS_PER_ITER,
        buffer_size=BUFFER_SIZE,
        min_buffer_size=MIN_BUFFER_SIZE,

        # Temperature schedule
        temp_threshold=TEMP_THRESHOLD,
        temp_final=TEMP_FINAL,

        # Curriculum learning
        curriculum_enabled=CURRICULUM_ENABLED,
        curriculum_initial_ratio=CURRICULUM_INITIAL_RATIO,
        curriculum_final_ratio=CURRICULUM_FINAL_RATIO,
        curriculum_decay_iterations=CURRICULUM_DECAY_ITERS,

        # Random opponent mixing
        random_opponent_enabled=RANDOM_OPP_ENABLED,
        random_opponent_initial_ratio=RANDOM_OPP_INITIAL_RATIO,
        random_opponent_final_ratio=RANDOM_OPP_FINAL_RATIO,
        random_opponent_decay_iterations=RANDOM_OPP_DECAY_ITERS,

        # Checkpointing
        checkpoint_dir=checkpoint_dir,
        checkpoint_every=CHECKPOINT_EVERY,

        # Eval
        eval_vs_random_games=50,

        # Wandb
        use_wandb=USE_WANDB,
        wandb_project=WANDB_PROJECT,
        wandb_run_name=run_name,
    )

print(f"Config factory ready")

Config factory ready


## Training Loops (Standard & Chimera)

In [ ]:
# Standard training (sequential board sizes)
def run_standard_training():
    """Train on each board size sequentially."""
    prev_trainer = None

    for stage_idx, (rows, cols, num_iters, mcts_sims) in enumerate(BOARD_STAGES):
        print(f"\n{'='*70}")
        print(f"STAGE {stage_idx+1}/{len(BOARD_STAGES)}: {rows}x{cols} board")
        print(f"Iterations: {num_iters} | MCTS sims: {mcts_sims}")
        print(f"Batch sizes: {GAMES_BATCH_SIZE} games, {TRAIN_BATCH_SIZE} train")
        print(f"Max moves/game: {rows * cols * 2}")
        print(f"{'='*70}\n")

        stage_dir = os.path.join(CHECKPOINT_BASE, "standard", f"{rows}x{cols}")
        os.makedirs(stage_dir, exist_ok=True)

        config = create_pmap_train_config(
            rows=rows,
            cols=cols,
            num_iterations=num_iters,
            num_simulations=mcts_sims,
            checkpoint_dir=stage_dir,
            run_name=f"standard_{rows}x{cols}_d{DEVICE_COUNT}",
        )

        trainer = AlphaZeroTrainer(config)

        # Resume from checkpoint if exists
        if os.path.exists(stage_dir):
            checkpoints = sorted([f for f in os.listdir(stage_dir) if f.endswith('.pkl')])
            if checkpoints:
                latest = os.path.join(stage_dir, checkpoints[-1])
                print(f"Resuming from: {latest}")
                trainer.load_checkpoint(latest)
                if trainer.iteration >= num_iters:
                    print(f"Stage complete, skipping...")
                    prev_trainer = trainer
                    continue

        # Train
        start = time.time()
        trainer.train()
        elapsed = time.time() - start

        print(f"\nStage {stage_idx+1} complete in {elapsed/60:.1f} min")
        prev_trainer = trainer

    print("\n" + "="*70)
    print("STANDARD TRAINING COMPLETE!")
    print("="*70)

    return prev_trainer


# Chimera training (shared backbone, all sizes together)
def run_chimera_training():
    """Train with shared backbone, separate policy heads per board size."""

    total_iters = sum(c[2] for c in BOARD_STAGES)
    mcts_sims = BOARD_STAGES[-1][3]

    # Use largest board for max_moves calculation
    max_rows, max_cols = max(CHIMERA_BOARD_SIZES, key=lambda x: x[0] * x[1])
    max_moves = max_rows * max_cols * 2

    print(f"\n{'='*70}")
    print(f"CHIMERA TRAINING")
    print(f"Board sizes: {CHIMERA_BOARD_SIZES}")
    print(f"Total iterations: {total_iters} | MCTS sims: {mcts_sims}")
    print(f"Batch sizes: {GAMES_BATCH_SIZE} games, {TRAIN_BATCH_SIZE} train")
    print(f"Max moves/game: {max_moves}")
    print(f"{'='*70}\n")

    chimera_dir = os.path.join(CHECKPOINT_BASE, "chimera")
    os.makedirs(chimera_dir, exist_ok=True)

    config = ChimeraConfig(
        board_sizes=CHIMERA_BOARD_SIZES,
        num_channels=NUM_CHANNELS,
        num_res_blocks=NUM_RES_BLOCKS,

        batch_size_games=GAMES_BATCH_SIZE,
        batch_size_train=TRAIN_BATCH_SIZE,
        max_moves_per_game=max_moves,
        temp_threshold=TEMP_THRESHOLD,
        num_simulations=mcts_sims,

        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        train_steps_per_iteration=TRAIN_STEPS_PER_ITER,
        buffer_size=BUFFER_SIZE,
        min_buffer_size=MIN_BUFFER_SIZE,

        num_iterations=total_iters,
        games_per_iteration=GAMES_BATCH_SIZE * 2,

        curriculum_enabled=CURRICULUM_ENABLED,
        curriculum_initial_ratio=CURRICULUM_INITIAL_RATIO,
        curriculum_final_ratio=CURRICULUM_FINAL_RATIO,
        curriculum_decay_iterations=CURRICULUM_DECAY_ITERS,

        checkpoint_dir=chimera_dir,
        checkpoint_every=CHECKPOINT_EVERY,

        use_wandb=USE_WANDB,
        wandb_project=WANDB_PROJECT,
        wandb_run_name=f"chimera_d{DEVICE_COUNT}",
    )

    trainer = ChimeraTrainer(config)

    start = time.time()
    trainer.train()
    elapsed = time.time() - start

    print(f"\nChimera training complete in {elapsed/60:.1f} min")

    return trainer


# Overnight two-stage: 11x9 -> 21x15 on same backbone
def run_overnight_11x9_training():
    """
    Two-stage overnight training:
    Stage 1: 200 iterations on 11x9 (fast, validate loop)
    Stage 2: 400 iterations on 21x15 (full board, same backbone)
    """
    print(f"\n{'='*70}")
    print(f"OVERNIGHT TWO-STAGE TRAINING")
    print(f"Stage 1: 200 iters on 11x9")
    print(f"Stage 2: 400 iters on 21x15 (same backbone)")
    print(f"{'='*70}\n")

    overnight_dir = os.path.join(CHECKPOINT_BASE, "overnight_11x9")
    os.makedirs(overnight_dir, exist_ok=True)

    # Stage 1: 11x9 only
    # max_moves_per_game=4000 (~6 hours at 5 sec/move)
    config = ChimeraConfig(
        board_sizes=((11, 9),),
        num_channels=64,
        num_res_blocks=20,

        batch_size_games=256,
        batch_size_train=512,
        max_moves_per_game=4000,
        temp_threshold=30,
        temp_final=0.1,
        num_simulations=16,

        learning_rate=1e-4,
        weight_decay=1e-4,
        train_steps_per_iteration=500,
        buffer_size=500_000,
        min_buffer_size=500,

        num_iterations=200,
        games_per_iteration=512,

        curriculum_enabled=False,

        league_enabled=True,
        league_pool_size=10,
        league_opponent_ratio=0.5,
        league_save_every=5,

        checkpoint_dir=overnight_dir,
        checkpoint_every=10,

        eval_vs_random_every=1,
        eval_vs_random_games=40,

        use_wandb=USE_WANDB,
        wandb_project=WANDB_PROJECT,
        wandb_run_name=f"overnight_2stage",
    )

    trainer = ChimeraTrainer(config)

    # Resume from checkpoint if exists
    if os.path.exists(overnight_dir):
        checkpoints = sorted([f for f in os.listdir(overnight_dir) if f.endswith('.pkl')])
        if checkpoints:
            latest = os.path.join(overnight_dir, checkpoints[-1])
            print(f"Resuming from: {latest}")
            trainer.load_checkpoint(latest)

    # Stage 1: Train on 11x9
    if trainer.iteration < 200:
        print(f"\n{'='*60}")
        print(f"STAGE 1: Training on 11x9 (iters {trainer.iteration} -> 200)")
        print(f"{'='*60}\n")
        start = time.time()
        trainer.train()
        elapsed = time.time() - start
        print(f"\nStage 1 complete in {elapsed/60:.1f} min")

    # Stage 2: Add 21x15 and continue training
    print(f"\n{'='*60}")
    print(f"STAGE 2: Adding 21x15, training 400 more iterations")
    print(f"{'='*60}\n")

    trainer.add_board_size(21, 15)
    trainer.config.num_iterations = 600  # 200 + 400
    trainer.config.num_simulations = 24  # Slightly more sims for bigger board

    start = time.time()
    trainer.train()
    elapsed = time.time() - start

    print(f"\nStage 2 complete in {elapsed/60:.1f} min")
    print(f"\nOvernight training complete!")

    return trainer


print(f"Training functions ready for mode: {TRAINING_MODE}")

In [ ]:
# Run training based on mode
if TRAINING_MODE == "standard":
    trainer = run_standard_training()
elif TRAINING_MODE == "chimera":
    trainer = run_chimera_training()
elif TRAINING_MODE == "overnight_11x9":
    trainer = run_overnight_11x9_training()
else:
    raise ValueError(f"Unknown training mode: {TRAINING_MODE}")


OVERNIGHT 11x9 BASELINE
Goal: Validate training by hitting 70%+ vs random



Chimera Training for Phutball
Board sizes: ['11x9']
Shared backbone: 64ch, 20 blocks
MCTS sims: 16
Training: 500 steps x 512 batch = 256,000 samples/iter
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]

Iteration 1/200
----------------------------------------
  Self-play: 19602 examples from 512 games (4.24 games/sec, 120.8s)
    11x9: 19602 ex | W1/W2/D=170/103/239 | moves=38.3, jump_seq=2.29, jump_len=1.32, adj=9.6% | buf=19602
  Training: 500 steps (28.6s) | p_loss: 4.3255, v_loss: 0.3955, kl: 1.3141, entropy: 4.371, v_pred: -0.002±0.261
  [Eval] vs random (40 games/board)...
    11x9: 19/40 wins (47.5%)
  [Eval] Avg win rate: 47.5%
  Iteration time: 175.5s

Iteration 2/200
----------------------------------------
  Self-play: 42860 examples from 512 games (3.42 games/sec, 149.8s)
    11x9: 42860 ex | W1/W2/D=188/234/90 | moves=83.7, jump_seq=1.73, jump_len=1.79, adj=13.3% | buf=62462
  Training: 500 steps (4.9s) | p_loss: 4.1947, v_loss: 0.4312, kl: 1.19

## Evaluation

In [ ]:
# Evaluate final model
if trainer:
    print("\nFinal evaluation vs random (100 games):")
    trainer.evaluate_vs_random(num_games=100)

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

if trainer and hasattr(trainer, 'metrics_history') and trainer.metrics_history:
    iters = [m['iteration'] for m in trainer.metrics_history]
    p_loss = [m['policy_loss'] for m in trainer.metrics_history]
    v_loss = [m['value_loss'] for m in trainer.metrics_history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(iters, p_loss)
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Policy Loss')
    axes[0].set_title('Policy Loss')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(iters, v_loss)
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Value Loss')
    axes[1].set_title('Value Loss')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No metrics to plot")

## Watch a Game

In [ ]:
# Watch model play
if trainer:
    state = reset(trainer.env_config)
    rng = jax.random.PRNGKey(42)

    print("=== Model vs Model ===")
    print(render_board(state))

    for move in range(50):
        rng, action_rng = jax.random.split(rng)

        action, policy, value = select_action(
            trainer.get_network_params(),
            action_rng,
            state,
            trainer.network,
            trainer.env_config,
            MCTSConfig(num_simulations=32, max_num_considered_actions=32),
            temperature=0.1,
        )

        state = step(state, jnp.array(action, dtype=jnp.int32), trainer.env_config)

        print(f"\nMove {move+1}: P{int(state.current_player)} action={action} value={value:.3f}")
        print(render_board(state))

        if state.terminated:
            print(f"\nGame over! Winner: Player {int(state.winner)}")
            break

## Quick Test

In [ ]:
# Quick sanity check (small settings)
def quick_test():
    print("Running quick test...")

    test_config = TrainConfig(
        rows=9,
        cols=7,
        num_iterations=2,
        num_channels=32,
        num_res_blocks=2,
        num_simulations=8,
        batch_size_games=max(8, DEVICE_COUNT * 4),
        batch_size_train=max(16, DEVICE_COUNT * 8),
        train_steps_per_iteration=5,
        buffer_size=500,
        min_buffer_size=50,
        checkpoint_every=999,
        curriculum_enabled=True,
        curriculum_initial_ratio=0.5,
    )

    test_trainer = AlphaZeroTrainer(test_config)
    test_trainer.train()

    print("Quick test passed!")
    return test_trainer

# Uncomment to run:
# quick_test()

## Download Checkpoints

In [ ]:
# List available checkpoints
import glob

print("Available checkpoints:")
for stage_dir in sorted(glob.glob(os.path.join(CHECKPOINT_BASE, "*"))):
    if os.path.isdir(stage_dir):
        ckpts = sorted(glob.glob(os.path.join(stage_dir, "*.pkl")))
        print(f"\n{os.path.basename(stage_dir)}: {len(ckpts)} checkpoints")
        for c in ckpts[-3:]:  # Last 3
            print(f"  - {os.path.basename(c)}")